# FTS ML Hyperparameter Optimization & Model Registry Workspace

Welcome to the interactive hyperparameter optimization and model registry workspace! This notebook shows you how to:
1. **Load database connections and settings** for FTS.
2. **Dynamically explore available model types** and configurations in the registry.
3. **Programmatically override and validate search configurations** before running optimization.
4. **Run Optuna hyperparameter search** using the core `hparam_search` engine.
5. **Visualize search results** using native Optuna Plotly graphs.
6. **Inspect candidate models** in a pandas DataFrame, and **promote** the best model to production.

### 1. Import Dependencies and Initialize DB Connections

In [1]:
import os
import yaml
import pandas as pd
import optuna
from datetime import datetime, timezone
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

from trading_bot.config import settings
from trading_bot.core.database import init_db, SessionLocal
from trading_bot.core.repository import MarketDataRepository, ModelRepository
from trading_bot.core.schemas import BarData

from nets.training.hparam_search import run_hparam_search, TRAINER_REGISTRY
from nets.models import NNTrainingConfig

# Set database URL dynamically to local dev.db
settings.DATABASE_URL = "sqlite+pysqlite:///../dev.db"
db_url = "sqlite:///../dev.db"
engine = create_engine(db_url, pool_pre_ping=True)
SessionLocal.configure(bind=engine)

# Initialize tables
init_db(extra_models=["trading_bot.core.models"], bind_engine=engine)
print("Database engine initialized. Available schemas prepared.")

Database engine initialized. Available schemas prepared.


### 2. Dynamic Model Registry Exploration

To adhere to the Open/Closed Principle (OCP), we retrieve the available model architectures dynamically from the codebase's central registry map. Adding any new model trainer or configuration in the core code automatically registers and exposes it here.

In [2]:
available_models = list(TRAINER_REGISTRY.keys())
print("Available model types in TRAINER_REGISTRY:", available_models)

for model_type, (trainer_cls, config_cls) in TRAINER_REGISTRY.items():
    print(f"\n- Model Type: '{model_type}'")
    print(f"  Trainer: {trainer_cls.__name__} & Config: {config_cls.__name__}")
    print(f"  Configurable Hyperparameters: {list(config_cls.model_fields.keys())}")

Available model types in TRAINER_REGISTRY: ['lstm', 'rnn', 'cnn', 'linear_regression', 'xgboost']

- Model Type: 'lstm'
  Trainer: LSTMTrainer & Config: LSTMConfig
  Configurable Hyperparameters: ['lookback_period', 'feature_cols', 'validation_split', 'embargo_pct', 'horizon', 'epochs', 'batch_size', 'learning_rate', 'optimizer', 'loss_fn', 'tensorboard_log_dir', 'early_stopping_patience', 'early_stopping_min_delta', 'clip_grad_norm', 'hidden_dim', 'num_layers', 'dropout', 'bidirectional']

- Model Type: 'rnn'
  Trainer: RNNTrainer & Config: RNNConfig
  Configurable Hyperparameters: ['lookback_period', 'feature_cols', 'validation_split', 'embargo_pct', 'horizon', 'epochs', 'batch_size', 'learning_rate', 'optimizer', 'loss_fn', 'tensorboard_log_dir', 'early_stopping_patience', 'early_stopping_min_delta', 'clip_grad_norm', 'hidden_dim', 'num_layers', 'dropout', 'nonlinearity']

- Model Type: 'cnn'
  Trainer: CNNTrainer & Config: CNNConfig
  Configurable Hyperparameters: ['lookback_period

### 3. Load & Override Search Configurations

We load the YAML search configuration file `configs/hparam_search.yaml` programmatically into a Python dictionary. This lets you inspect the configuration and perform overrides directly in cell code for full reproducibility, bypassing complex Jupyter UI states.

In [3]:
CHOSEN_MODEL_TYPE = "lstm"

config_path = f"../configs/train/BTCUSDT/{CHOSEN_MODEL_TYPE}_hparam_search.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Current YAML Configuration:")
print(yaml.dump(config, default_flow_style=False))

Current YAML Configuration:
direction: minimize
feature_cols:
- close
interval: 30m
lookback_period: 20
market_id: BTC/USDT
model_type: lstm
n_trials: 10
search_space:
  batch_size:
    high: 32
    low: 32
    type: int
  bidirectional:
    choices:
    - false
    type: categorical
  dropout:
    high: 0.1
    low: 0.1
    type: float
  epochs:
    high: 10
    low: 10
    type: int
  hidden_dim:
    high: 64
    low: 16
    type: int
  learning_rate:
    high: 0.01
    log: true
    low: 0.0001
    type: float
  loss_fn:
    choices:
    - mse
    type: categorical
  num_layers:
    high: 1
    low: 1
    type: int
  optimizer:
    choices:
    - adam
    type: categorical
study_name: lstm_btc_optimization



### 4. Configuration Validation

We use the Pydantic schemas (LSP/ISP validation) to dynamically validate that all hyperparameter keys specified under `search_space` match expected parameter fields in the model configuration class and core trainer configurations.

In [4]:
# Validate loaded configurations dynamically
model_type = config["model_type"]
if model_type not in TRAINER_REGISTRY:
    raise ValueError(f"Model type '{model_type}' is invalid. Supported: {available_models}")

trainer_cls, config_cls = TRAINER_REGISTRY[model_type]
print(f"Validating search space params against {config_cls.__name__} & NNTrainingConfig...")

model_fields = set(config_cls.model_fields.keys())
nn_fields = set(NNTrainingConfig.model_fields.keys())
all_valid_fields = model_fields.union(nn_fields)

search_space = config.get("search_space") or {}
for param in search_space:
    if param not in all_valid_fields:
        print(f"⚠️  WARNING: Parameter '{param}' is not defined in the core model configuration classes.")
    else:
        print(f"  - Parameter '{param}' validated successfully.")

Validating search space params against LSTMConfig & NNTrainingConfig...
  - Parameter 'learning_rate' validated successfully.
  - Parameter 'hidden_dim' validated successfully.
  - Parameter 'epochs' validated successfully.
  - Parameter 'batch_size' validated successfully.
  - Parameter 'num_layers' validated successfully.
  - Parameter 'dropout' validated successfully.
  - Parameter 'bidirectional' validated successfully.
  - Parameter 'optimizer' validated successfully.
  - Parameter 'loss_fn' validated successfully.


### 4.1. Visualize Raw and Preprocessed Training Data

We load the historical bar data from the database using the same repository queries that the training pipeline runs, apply the log return preprocessing, and visualize both the raw price series and the stationary log returns input features.

In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from trading_bot.core.dataset import DatasetBuilder
from trading_bot.core.transforms import FeaturePipeline, LogReturnTransform, RatioTransform
import numpy as np

# 1. Fetch raw bar data using core repository
market_id = config["market_id"]
interval = config.get("interval", "30m")

with SessionLocal() as db:
    market_repo = MarketDataRepository(db)
    raw_bars = market_repo.get_bars(market_id, interval=interval)

# 2. Convert to DataFrame
df_viz = pd.DataFrame([{
    "timestamp": b.timestamp,
    "open": b.open,
    "high": b.high,
    "low": b.low,
    "close": b.close,
    "volume": b.volume
} for b in raw_bars])

# 3. Build and execute FeaturePipeline (representing the 5 features from quant study plan)
pipeline = FeaturePipeline(transforms=[
    LogReturnTransform(col_idx=3),                              # 0: Close Return
    RatioTransform(num_idx=1, den_idx=3),                       # 1: High Ratio
    RatioTransform(num_idx=2, den_idx=3),                       # 2: Low Ratio
    RatioTransform(num_idx=0, den_idx=3),                       # 3: Open Ratio
    LogReturnTransform(col_idx=4),                              # 4: Volume Return
])

# Extract matrix of raw OHLCV features (columns: open=0, high=1, low=2, close=3, volume=4)
matrix_raw = df_viz[["open", "high", "low", "close", "volume"]].values
features = pipeline.fit_transform(matrix_raw)

# Since LogReturnTransform reduces length by 1, FeaturePipeline slices all columns to match.
# Insert a row of NaNs at the beginning to align with df_viz timestamps
features_aligned = np.insert(features, 0, np.nan, axis=0)

df_viz["close_return"] = features_aligned[:, 0]
df_viz["high_ratio"] = features_aligned[:, 1]
df_viz["low_ratio"] = features_aligned[:, 2]
df_viz["open_ratio"] = features_aligned[:, 3]
df_viz["volume_return"] = features_aligned[:, 4]

# 4. Plot original price and the 5 stationary features using Plotly
fig_viz = make_subplots(
    rows=6, cols=1, shared_xaxes=True,
    subplot_titles=(
        f"Raw Close Price ({market_id})", 
        "Close Log Return", 
        "High/Close Log Ratio", 
        "Low/Close Log Ratio", 
        "Open/Close Log Ratio", 
        "Volume Log Change"
    )
)

fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["close"], name="Close Price", line=dict(color="#2196F3")), row=1, col=1)
fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["close_return"], name="Close Return", line=dict(color="#FF9800")), row=2, col=1)
fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["high_ratio"], name="High Ratio", line=dict(color="#4CAF50")), row=3, col=1)
fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["low_ratio"], name="Low Ratio", line=dict(color="#F44336")), row=4, col=1)
fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["open_ratio"], name="Open Ratio", line=dict(color="#9C27B0")), row=5, col=1)
fig_viz.add_trace(go.Scatter(x=df_viz["timestamp"], y=df_viz["volume_return"], name="Volume Return", line=dict(color="#795548")), row=6, col=1)

fig_viz.update_layout(height=1200, title_text=f"Multi-Dimensional Stationary Feature Pipeline Overview", showlegend=False)
fig_viz.show()


### 5. Execute Hyperparameter Optimization

We invoke `run_hparam_search` directly using our updated config file. Optuna will evaluate different parameter sets, train candidates, log metrics to TensorBoard, and register model metadata to our registry.

In [6]:
print(f"Starting Optuna study '{config['study_name']}' ({config['n_trials']} trials)...\n")
run_hparam_search(config_path)
print("\nHyperparameter optimization study complete!")

Starting Optuna study 'lstm_btc_optimization' (10 trials)...



[I 2026-06-29 00:28:20,910] Using an existing study with name 'lstm_btc_optimization' instead of creating a new one.
/home/alfred/github/fts/src/plugins/nets/training/abc.py:481: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0629 00:28:24.290000 107325 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0629 00:28:24.292000 107325 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0629 00:28:24.292000 107325 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `SimpleLSTM([...]` with `torch.export.export(..., strict=False)`...


/home/alfred/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: UserWarning: The tensor attributes self.lstm._flat_weights[0], self.lstm._flat_weights[1], self.lstm._flat_weights[2], self.lstm._flat_weights[3] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `SimpleLSTM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


...

[torch.onnx] Obtain model graph for `SimpleLSTM([...]` with `torch.export.export(..., strict=False)`...


/home/alfred/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/contextlib.py:144: UserWarning: The tensor attributes self.lstm._flat_weights[0], self.lstm._flat_weights[1], self.lstm._flat_weights[2], self.lstm._flat_weights[3] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  next(self.gen)


[torch.onnx] Obtain model graph for `SimpleLSTM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/alfred/.local/share/uv/python/cpython-3.12.10-linux-x86_64-gnu/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
[I 2026-06-29 00:28:54,081] Trial 54 finished with value: 6.6762027017830405e-06 and parameters: {'learning_rate': 0.0038724770708502914, 'hidden_dim': 45, 'epochs': 10, 'batch_size': 32, 'num_layers': 1, 'dropout': 0.1, 'bidirectional': False, 'optimizer': 'adam', 'loss_fn': 'mse'}. Best is trial 45 with value: 5.776538273494225e-06.


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Best trial: 45 with loss 5.776538273494225e-06

Hyperparameter optimization study complete!


### 6. Plot Optuna Optimization Visualizations

We load the Optuna study programmatically from SQLite and render native interactive plots using Optuna's native Plotly backend.

In [7]:
optuna_storage = settings.DATABASE_URL.replace("sqlite+pysqlite://", "sqlite://")
try:
    study = optuna.load_study(study_name=config["study_name"], storage=optuna_storage)
    print(f"Loaded study '{study.study_name}' containing {len(study.trials)} trials.")
    print(f"Best trial: {study.best_trial.number} | Best Value: {study.best_value}")

    # Render native Plotly plots
    fig1 = optuna.visualization.plot_optimization_history(study)
    fig1.show()

    if len(study.trials) > 1:
        fig2 = optuna.visualization.plot_param_importances(study)
        fig2.show()
        
        fig3 = optuna.visualization.plot_slice(study)
        fig3.show()
except Exception as e:
    print("Could not load or plot Optuna study visualizations:", e)

Loaded study 'lstm_btc_optimization' containing 55 trials.
Best trial: 45 | Best Value: 5.776538273494225e-06


### 7. Inspect Model Registry Candidates

We use pandas to fetch all registered candidates from our database model registry table (`model_registry`). This provides a tabular dashboard of all run trials, hyperparameters, and validation metrics.

In [8]:
from nets.training.hparam_search import get_scored_models

with SessionLocal() as db:
    df_showcase = get_scored_models(
        db,
        model_type=config["model_type"],
        market_id=config["market_id"],
        interval=config.get("interval", "30m")
    )

# Display showcase dataframe
display_cols = [
    "model_id", "model_type", "market_id", "interval", 
    "val_loss", "ic", "directional_accuracy", "composite_score", "status", "created_at"
]
df_showcase[display_cols].head(15)

,model_id,model_type,market_id,interval,val_loss,ic,directional_accuracy,composite_score,status,created_at
29,model_lstm_btcusdt_30m_20260628_023716_2299dd,lstm,BTC/USDT,30m,0.000006,0.137865,0.584615,0.951104,candidate,2026-06-28 02:37:16
48,755008be4b7a,lstm,BTC/USDT,30m,0.000006,0.107265,0.584615,0.913764,candidate,2026-06-29 03:28:44
7,model_lstm_btcusdt_30m_20260628_020638_ea21ed,lstm,BTC/USDT,30m,0.000006,0.178350,0.548718,0.907437,candidate,2026-06-28 02:06:38
40,ef154c7c47ea,lstm,BTC/USDT,30m,0.000006,0.132405,0.553846,0.858187,candidate,2026-06-28 03:35:04
42,8dfdcbc31e3e,lstm,BTC/USDT,30m,0.000006,0.102489,0.548718,0.818248,candidate,2026-06-29 03:28:25
21,model_lstm_btcusdt_30m_20260628_023315_2c0070,lstm,BTC/USDT,30m,0.000006,0.079114,0.558974,0.814965,candidate,2026-06-28 02:33:15
8,model_lstm_btcusdt_30m_20260628_020642_3a0d72,lstm,BTC/USDT,30m,0.000006,0.129681,0.533333,0.811160,candidate,2026-06-28 02:06:42
26,model_lstm_btcusdt_30m_20260628_023709_bdda44,lstm,BTC/USDT,30m,0.000006,0.148982,0.517949,0.793313,candidate,2026-06-28 02:37:09
24,model_lstm_btcusdt_30m_20260628_023703_0912fa,lstm,BTC/USDT,30m,0.000006,0.090652,0.533333,0.764938,candidate,2026-06-28 02:37:03
35,260f7f2df026,lstm,BTC/USDT,30m,0.000006,0.130136,0.512821,0.758861,candidate,2026-06-28 03:34:45


### 8. Programmatic Model Promotion

To promote a model to production status, copy the `model_id` from the DataFrame above and paste it below. The repository will demote any active production model sharing the same signature and promote the selected candidate.

In [9]:
# --- ENTER THE MODEL ID TO PROMOTE ---
model_id_to_promote = ""  # e.g., "model_lstm_btc_usd_..."

if model_id_to_promote:
    with SessionLocal() as db:
        repo = ModelRepository(db)
        repo.promote_to_production(model_id_to_promote)
        db.commit()
    print(f"Model '{model_id_to_promote}' successfully promoted to PRODUCTION status.")
    
    # Display status verification
    with SessionLocal() as db:
        df_verify = pd.read_sql(f"SELECT model_id, status, onnx_path FROM model_registry WHERE model_id='{model_id_to_promote}'", db.bind)
    print("\nUpdated database status:")
    print(df_verify)
else:
    print("Please copy/paste a valid 'model_id' into 'model_id_to_promote' to promote it.")

Please copy/paste a valid 'model_id' into 'model_id_to_promote' to promote it.
